In [1]:
# Cell 1: Install dependencies
!pip install -q polars pyarrow

In [2]:
# Cell 2: Inspect input files
from pathlib import Path

for p in sorted(Path("/kaggle/input").glob("**/*")):
    if p.is_file():
        print(p, round(p.stat().st_size / 1024 / 1024, 2), "MB")

/kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/build_kg_kaggle.py 0.03 MB
/kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/chunks.parquet 7111.26 MB
/kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/metadata.parquet 19.61 MB
/kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/relationships.parquet 0.68 MB


In [3]:
# Cell 3: Locate files
from pathlib import Path

def find_one(name: str) -> Path:
    matches = list(Path("/kaggle/input").glob(f"**/{name}"))
    if not matches:
        raise FileNotFoundError(f"Cannot find {name} under /kaggle/input")
    if len(matches) > 1:
        print(f"WARNING: multiple {name} files found:")
        for m in matches:
            print(" ", m)
        print("Using:", matches[0])
    return matches[0]

metadata = find_one("metadata.parquet")
relationships = find_one("relationships.parquet")
chunks = find_one("chunks.parquet")
script = find_one("build_kg_kaggle.py")

print("metadata:", metadata)
print("relationships:", relationships)
print("chunks:", chunks)
print("script:", script)

metadata: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/metadata.parquet
relationships: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/relationships.parquet
chunks: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/chunks.parquet
script: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/build_kg_kaggle.py


In [4]:
# Cell 4: Quick row counts without loading full data
import pyarrow.parquet as pq

for name, path in {
    "metadata": metadata,
    "relationships": relationships,
    "chunks": chunks,
}.items():
    pf = pq.ParquetFile(path)
    print(
        name,
        "rows=", pf.metadata.num_rows,
        "cols=", len(pf.schema_arrow.names),
        "size_mb=", round(path.stat().st_size / 1024 / 1024, 2),
    )

metadata rows= 353746 cols= 18 size_mb= 19.61
relationships rows= 138656 cols= 3 size_mb= 0.68
chunks rows= 7476806 cols= 29 size_mb= 7111.26


In [5]:
# Cell 5: Run KG build
# Output should be:
# /kaggle/working/neo4j_imports
# /kaggle/working/neo4j_imports.zip

!python "{script}"

2026-07-04 01:34:58,334 - INFO - Configured Paths:
2026-07-04 01:34:58,334 - INFO -   metadata: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/metadata.parquet
2026-07-04 01:34:58,334 - INFO -   chunks: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/chunks.parquet
2026-07-04 01:34:58,334 - INFO -   relationships: /kaggle/input/datasets/nguyenlehoangtrung/vietlegal-kg-build/relationships.parquet
2026-07-04 01:34:58,334 - INFO -   output_dir: /kaggle/working/neo4j_imports
2026-07-04 01:34:58,334 - INFO -   zip_output: /kaggle/working/neo4j_imports.zip
2026-07-04 01:34:58,334 - INFO - Loading Parquet files...
2026-07-04 01:34:58,930 - INFO - Loaded metadata: 353746 rows
2026-07-04 01:35:01,231 - INFO - Loaded chunks (lightweight columns only): 7476806 rows (columns loaded: ['doc_id', 'chunk_id', 'chunk_type', 'section_path', 'section_path_json', 'procedure_name', 'token_estimate', 'quality_flags_json', 'parent_id'])
2026-07-04 01:35:01,250 - INFO - Loaded relat

In [6]:
# Cell 6: Inspect output size
from pathlib import Path

out_dir = Path("/kaggle/working/neo4j_imports")
zip_path = Path("/kaggle/working/neo4j_imports.zip")

print("Output dir exists:", out_dir.exists())
print("Zip exists:", zip_path.exists())

if out_dir.exists():
    for p in sorted(out_dir.glob("*")):
        if p.is_file():
            print(p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

if zip_path.exists():
    print("ZIP:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

Output dir exists: True
Zip exists: True
art_art_next_edges.csv 347.86 MB
art_art_parent_edges.csv 191.02 MB
articles.csv 1436.24 MB
authorities.csv 0.26 MB
cleaned_relationships.parquet 0.64 MB
doc_art_edges.csv 293.85 MB
doc_auth_edges.csv 17.09 MB
doc_doc_edges.csv 9.11 MB
doc_sector_edges.csv 23.38 MB
doc_type_edges.csv 16.14 MB
documents.csv 136.65 MB
legal_types.csv 0.0 MB
sectors.csv 0.18 MB
ZIP: 386.65 MB


In [7]:
# Cell 7: Lightweight audit for build_kg_kaggle output
import csv
from pathlib import Path
from collections import Counter

out_dir = Path("/kaggle/working/neo4j_imports")

required_files = [
    "documents.csv",
    "articles.csv",
    "legal_types.csv",
    "authorities.csv",
    "sectors.csv",
    "doc_doc_edges.csv",
    "doc_art_edges.csv",
    "art_art_next_edges.csv",
    "art_art_parent_edges.csv",
    "doc_type_edges.csv",
    "doc_auth_edges.csv",
    "doc_sector_edges.csv",
]

missing = [name for name in required_files if not (out_dir / name).exists()]
print("Missing files:", missing)

def count_rows(path: Path) -> int:
    with path.open("rb") as f:
        return max(0, sum(block.count(b"\n") for block in iter(lambda: f.read(1024 * 1024), b"")) - 1)

for name in required_files:
    path = out_dir / name
    if path.exists():
        print(name, count_rows(path))

Missing files: []
documents.csv 353746
articles.csv 7476806
legal_types.csv 37
authorities.csv 1758
sectors.csv 1583
doc_doc_edges.csv 120107
doc_art_edges.csv 7476806
art_art_next_edges.csv 7123141
art_art_parent_edges.csv 3971734
doc_type_edges.csv 353744
doc_auth_edges.csv 353536
doc_sector_edges.csv 593983


In [8]:
# Cell 8: Check legal relation distribution and raw label preservation
import csv
from pathlib import Path
from collections import Counter

edge_path = Path("/kaggle/working/neo4j_imports/doc_doc_edges.csv")

rel_counts = Counter()
missing_raw = 0
headers = []

with edge_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames or []
    for row in reader:
        rel_counts[row.get(":TYPE", "")] += 1
        raw = row.get("raw_label") or row.get("raw_relationship") or ""
        if not raw.strip():
            missing_raw += 1

print("headers:", headers)
print("relation counts:", dict(rel_counts))
print("missing raw label/raw relationship:", missing_raw)

allowed = {"AMENDS", "CITES", "REPEALS", "GUIDELINES"}
unknown = set(rel_counts) - allowed
print("unknown relation types:", unknown)

headers: [':START_ID(Document)', ':END_ID(Document)', ':TYPE', 'weight:double', 'raw_relationship', 'raw_label']
relation counts: {'AMENDS': 4305, 'CITES': 107597, 'GUIDELINES': 5493, 'REPEALS': 2712}
missing raw label/raw relationship: 0
unknown relation types: set()


In [9]:
# Cell 9: Create a small manifest for download/debug
import json
from pathlib import Path

out_dir = Path("/kaggle/working/neo4j_imports")
manifest = {}

for p in sorted(out_dir.glob("*")):
    if p.is_file():
        manifest[p.name] = {
            "size_mb": round(p.stat().st_size / 1024 / 1024, 2),
        }

Path("/kaggle/working/kg_build_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "art_art_next_edges.csv": {
    "size_mb": 347.86
  },
  "art_art_parent_edges.csv": {
    "size_mb": 191.02
  },
  "articles.csv": {
    "size_mb": 1436.24
  },
  "authorities.csv": {
    "size_mb": 0.26
  },
  "cleaned_relationships.parquet": {
    "size_mb": 0.64
  },
  "doc_art_edges.csv": {
    "size_mb": 293.85
  },
  "doc_auth_edges.csv": {
    "size_mb": 17.09
  },
  "doc_doc_edges.csv": {
    "size_mb": 9.11
  },
  "doc_sector_edges.csv": {
    "size_mb": 23.38
  },
  "doc_type_edges.csv": {
    "size_mb": 16.14
  },
  "documents.csv": {
    "size_mb": 136.65
  },
  "legal_types.csv": {
    "size_mb": 0.0
  },
  "sectors.csv": {
    "size_mb": 0.18
  }
}


In [10]:
# Cell 10: Ensure zip exists
# build_kg_kaggle.py should already create this, but recreate if needed.

from pathlib import Path
import zipfile

out_dir = Path("/kaggle/working/neo4j_imports")
zip_path = Path("/kaggle/working/neo4j_imports.zip")

if not zip_path.exists():
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in out_dir.glob("*.csv"):
            zf.write(p, arcname=p.name)

print("Download:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

Download: /kaggle/working/neo4j_imports.zip
Size MB: 386.65
